# Student Dropout Prediction - Model Training

This notebook trains the final MVP pipeline using the fixed 10 enrollment/background features. It compares Logistic Regression and Random Forest, then saves the best final pipeline and metadata.

In [ ]:
# Import libraries used for modeling.
from pathlib import Path
import json

import joblib
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Load Processed Data

The processed dataset already contains only Graduate/Dropout records and the fixed MVP feature scope.

In [ ]:
# Resolve paths.
current_path = Path.cwd()

if (current_path / "data" / "processed" / "processed.csv").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parent

PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "processed.csv"
FEATURE_CONFIG_PATH = PROJECT_ROOT / "app" / "feature_config.json"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_PATH = MODEL_DIR / "final_mvp_model.pkl"
MVP_FEATURES_PATH = MODEL_DIR / "mvp_features.json"
MODEL_METADATA_PATH = MODEL_DIR / "model_metadata.json"

print("Processed data path:", PROCESSED_DATA_PATH)
print("Model path:", MODEL_PATH)
print("Metadata path:", MODEL_METADATA_PATH)

In [ ]:
# Load data and config.
df_processed = pd.read_csv(PROCESSED_DATA_PATH)

with open(FEATURE_CONFIG_PATH, "r", encoding="utf-8") as file:
    feature_config = json.load(file)

candidate_mvp_features = feature_config["features"]

print("Processed dataset shape:", df_processed.shape)
print("Target distribution:")
print(df_processed["Target"].value_counts().sort_index())

df_processed.head()

## Define Features

`Age at enrollment` is the only continuous feature. All other selected features are treated as categorical-like encoded fields.

In [ ]:
# Define feature matrix and target.
X = df_processed[candidate_mvp_features].copy()
y = df_processed["Target"].copy()

# Define continuous features.
continuous_features = [
    "Age at enrollment"
]

# Define categorical-like features.
categorical_features = [
    col for col in X.columns
    if col not in continuous_features
]

print("Continuous features:", continuous_features)
print("Categorical features:", categorical_features)

## Train, Validation, and Test Split

The validation set is used to choose between the two models. The test set is kept for final evaluation.

In [ ]:
# Split data into train, validation, and test sets.
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=42,
    stratify=y_train_full
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("Test shape:", X_test.shape)

## Build Preprocessor and Models

OneHotEncoder handles categorical fields, while StandardScaler scales the continuous age feature. Logistic Regression is the interpretable baseline; Random Forest is the non-linear comparison model.

In [ ]:
# Build preprocessing pipeline.
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("continuous", StandardScaler(), continuous_features)
    ],
    remainder="drop"
)

# Define classification models.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
}

models

## Model Comparison

Models are compared on validation metrics, with F1-score used as the main selection metric because Dropout is the positive risk class.

In [ ]:
# Train and evaluate each model on the validation set.
def evaluate_model(name, pipeline, X_eval, y_eval):
    y_pred = pipeline.predict(X_eval)
    y_proba = pipeline.predict_proba(X_eval)[:, 1]

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_eval, y_pred),
        "Precision": precision_score(y_eval, y_pred, zero_division=0),
        "Recall": recall_score(y_eval, y_pred, zero_division=0),
        "F1-Score": f1_score(y_eval, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, y_proba)
    }

validation_results = []
fitted_validation_pipelines = {}

for model_name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", clone(estimator))
    ])

    pipeline.fit(X_train, y_train)
    fitted_validation_pipelines[model_name] = pipeline
    validation_results.append(evaluate_model(model_name, pipeline, X_valid, y_valid))

validation_results_df = (
    pd.DataFrame(validation_results)
    .sort_values(["F1-Score", "ROC-AUC"], ascending=False)
    .reset_index(drop=True)
)

validation_results_df

In [ ]:
# Select the best model from validation performance.
best_model_name = validation_results_df.loc[0, "Model"]

print("Best model:", best_model_name)

## Final Model Evaluation

The best estimator is retrained on train + validation data, then evaluated once on the test set.

In [ ]:
# Train final model using train + validation data.
final_estimator = clone(models[best_model_name])

final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", final_estimator)
])

final_pipeline.fit(X_train_full, y_train_full)

final_pipeline

In [ ]:
# Evaluate the final model on the test set.
final_evaluation = evaluate_model("Final Test Evaluation", final_pipeline, X_test, y_test)
final_evaluation_df = pd.DataFrame([final_evaluation])

y_test_pred = final_pipeline.predict(X_test)

print("Confusion matrix:")
print(confusion_matrix(y_test, y_test_pred))
print("\nClassification report:")
print(classification_report(y_test, y_test_pred, target_names=["Graduate", "Dropout"]))

final_evaluation_df

## Feature Importance for Interpretation

Feature importance is reported only for interpretation. It is not used to select or remove MVP features.

In [ ]:
# Show feature importance when the selected final model exposes it.
if hasattr(final_pipeline.named_steps["model"], "feature_importances_"):
    transformed_feature_names = final_pipeline.named_steps["preprocessor"].get_feature_names_out()
    importance_df = (
        pd.DataFrame({
            "feature": transformed_feature_names,
            "importance": final_pipeline.named_steps["model"].feature_importances_
        })
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
else:
    importance_df = pd.DataFrame(columns=["feature", "importance"])

importance_df.head(20)

## Save Model Artifacts

The saved metadata uses the fixed 10 MVP features from config, not a feature-selection output.

In [ ]:
# Save model, fixed MVP features, and metadata.
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(final_pipeline, MODEL_PATH)

with open(MVP_FEATURES_PATH, "w", encoding="utf-8") as file:
    json.dump(candidate_mvp_features, file, indent=4)

model_metadata = {
    "model_name": "Final Student Dropout Prediction Model",
    "base_model": best_model_name,
    "target_mapping": {
        "Graduate": 0,
        "Dropout": 1
    },
    "positive_class": "Dropout",
    "positive_class_value": 1,
    "mvp_feature_count": len(candidate_mvp_features),
    "mvp_features": candidate_mvp_features,
    "continuous_features": continuous_features,
    "categorical_features": categorical_features,
    "selection_metric": "F1-Score",
    "validation_results": validation_results_df.to_dict(orient="records"),
    "evaluation": final_evaluation_df.to_dict(orient="records")
}

with open(MODEL_METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(model_metadata, file, indent=4)

print("Model saved:", MODEL_PATH.exists())
print("MVP features saved:", MVP_FEATURES_PATH.exists())
print("Model metadata saved:", MODEL_METADATA_PATH.exists())

model_metadata

## Modeling Summary

- The final experiment uses fixed enrollment/background MVP features from start to finish.
- Logistic Regression provides a simple interpretable baseline.
- Random Forest provides a stronger non-linear comparison for categorical-heavy tabular data.
- Feature importance is interpretive only; it does not change the fixed MVP scope.